In [7]:
import pandas as pd, numpy as np
from pprint import pprint
import csv, sqlite3, json

from video_game_retention.api import (
    get_steam_game_details,
    get_steam_player_history,
    get_top_games
    )
from video_game_retention.config import (
    PROJECT_ROOT,
    DATABASE_PATH
    )

from video_game_retention.database import (
    init_database,
    save_game_detail,
    save_player_history
    )

## Exploring Manual API Fetch 
Fetching data via SteamIdx to check API connection. Taking a look at the values and variable types.

In [ ]:
# Grand Theft Auto V Legacy (271590)
game = get_steam_game_details(271590)
history = get_steam_player_history(271590)

points = history["points"]

print("Game:", game["name"])
print("Release date:", game["releaseDate"])
print("History resolution:", history["resolution"])
print("Observations:", len(points))

if points:
    print("Earliest observation:", min(p["at"] for p in points))
    print("Latest observation:", max(p["at"] for p in points))

In [ ]:
# variable type
for key, value in game.items():
    print(f"{key}: {type(value).__name__}")

In [ ]:
pprint(game)

In [ ]:
app_ids = [
    730,      # Counter-Strike 2
    1086940,  # Baldur's Gate 3
    1091500,  # Cyberpunk 2077
    271590,
]

results = []

for app_id in app_ids:
    game = get_steam_game_details(app_id)
    history = get_steam_player_history(app_id, "all")

    points = history.get("points", [])

    results.append({
        "app_id": app_id,
        "name": game["name"],
        "release_date": game["releaseDate"],
        "first_observation": min(
            (p["at"] for p in points),
            default=None,
        ),
        "last_observation": max(
            (p["at"] for p in points),
            default=None,
        ),
        "observations": len(points),
        "resolution": history.get("resolution"),
    })

df = pd.DataFrame(results)
display(df)

Earliest observation seems to be very recent. Might mean that we will lack numbers of observation in our data.

## SteamIDX and Fronkon Database Connection.
Checking connection between the data from the SteamIDX API vs the fronkon dataset. Check if ID correspond to each other, and it does.  

In [ ]:
# get the top 50 games from SteamIDX -- can me change later
top_games = get_top_games(100)

print(type(top_games))
print(len(top_games))

In [ ]:
pprint(top_games[:10])

In [ ]:
# load fronkon 
csv_path = f"{PROJECT_ROOT}/data/raw/games.csv"

with open(csv_path, encoding="utf-8", newline="") as f:
    header = next(csv.reader(f))

corrected_header = (
    header[:7]
    + ["Discount", "DLC count"]
    + header[8:]
)

fronkon_df = pd.read_csv(
    csv_path,
    header=0,
    names=corrected_header,
    index_col=False
)

fronkon_ids = (
    pd.to_numeric(fronkon_df['AppID'], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna(how='all').astype(int).drop_duplicates().tolist()
)

In [ ]:
top_df = pd.DataFrame(top_games)

top_df["in_fronkon"] = top_df["appId"].isin(fronkon_ids)

print("Already in FronkonGames:", top_df["in_fronkon"].sum())
print("Missing from FronkonGames:", (~top_df["in_fronkon"]).sum())

In [ ]:
# GTA V legacy in?
print("In FronkonGames:", 271590 in fronkon_ids)

In [ ]:
print("In SteamIDX Top 50:", 271590 in top_df["appId"].astype(int).tolist())

In [ ]:
top_df.head().sort_values(by="allTimePeak", ascending=False)

In [ ]:
top_100_id = top_df.sort_values(by = "allTimePeak", ascending=False)["appId"].to_list()

In [ ]:
top_100_id

In [ ]:
top_df[~top_df["in_fronkon"]][
    ["appId", "name"]
]

## Database
Testing database connection

In [ ]:
init_database()
print(DATABASE_PATH)

In [8]:
with sqlite3.connect(DATABASE_PATH) as conn:
    tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        conn,
    )

tables

,name
0,games
1,player_history


### Testing manual insert of games table

In [ ]:
game = get_steam_game_details(620)

with sqlite3.connect(DATABASE_PATH) as conn:
    conn.execute(
        """
        INSERT INTO games (app_id, name, details_json)
        VALUES (?, ?, ?)
        ON CONFLICT(app_id) DO UPDATE SET
            name = excluded.name,
            details_json = excluded.details_json,
            collected_at = CURRENT_TIMESTAMP
        """,
        (
            game["appId"],
            game["name"],
            json.dumps(game),
        ),
    )

print("Saved:", game["name"])

In [ ]:
with sqlite3.connect(DATABASE_PATH) as conn:
    saved_game = pd.read_sql_query(
        """
        SELECT app_id, name, collected_at, details_json
        FROM games
        WHERE app_id = ?
        """,
        conn,
        params=(570,),
    )

saved_game

In [ ]:
with sqlite3.connect(DATABASE_PATH) as conn:
    row = conn.execute(
        "SELECT details_json FROM games WHERE app_id = ?",
        (620,),
    ).fetchone()

saved_details = json.loads(row[0])

print(saved_details["name"])
print(saved_details["genres"])
print(saved_details["reviews"])

### Testing player_history table

In [ ]:
history = get_steam_player_history(620, range_="all")

In [ ]:
print(type(history))
print(history if isinstance(history, dict) else history[:2])

In [ ]:
rows = [
    (history["appId"], point["at"], point["players"])
    for point in history["points"]
]

with sqlite3.connect(DATABASE_PATH) as conn:
    conn.executemany(
        """
        INSERT INTO player_history (app_id, observed_at, players)
        VALUES (?, ?, ?)
        ON CONFLICT(app_id, observed_at) DO UPDATE SET
            players = excluded.players
        """,
        rows,
    )

print(f"Saved {len(rows)} player-history observations.")

In [ ]:
with sqlite3.connect(DATABASE_PATH) as conn:
    saved_history = pd.read_sql_query(
        """
        SELECT app_id, observed_at, players
        FROM player_history
        WHERE app_id = ?
        ORDER BY observed_at
        """,
        conn,
        params=(620,),
    )

saved_history.head()

In [ ]:
saved_history.tail()

In [ ]:
print("API observations:", len(history["points"]))
print("Database observations:", len(saved_history))

save_player_history(history)

In [ ]:
top_games = get_top_games(200)

print("Games retrieved:", len(top_games))
print("Unique App IDs:", len({game["appId"] for game in top_games}))

In [ ]:
for i in range(95, 105):
    game = top_games[i]
    print(i + 1, game["appId"], game["name"])

In [ ]:
with sqlite3.connect(DATABASE_PATH) as conn:
    result = conn.execute(
        "SELECT COUNT(*) FROM games"
    ).fetchone()

print(f"Total games in database: {result[0]}")

In [ ]:
with sqlite3.connect(DATABASE_PATH) as conn:
    result = conn.execute(
        "SELECT COUNT(*) FROM player_history"
    ).fetchone()

print(f"games players in database: {result[0]}")

{(620, '2026-09-07T00:00:00.000Z', 2660)}
